In [3]:
from pandas import DataFrame
import torch
import numpy as np
from dataLoader import *
from MultiOutputTuner import *
from pathlib import Path
from BaselineRegressor import BaselineRegressor
from sklearn.metrics import mean_squared_error
from MultiOutputTuner import MultiOutputTuner
from sklearn.preprocessing import StandardScaler

In [14]:
X_train = load_msi_data(HSI_SATELLITE_DIR)
y_train = gt_train_df[column_names].values

baseline = BaselineRegressor()
baseline.fit(X_train, y_train)

y_baseline = baseline.predict(X_train)
mse_baseline = mean_squared_error(y_train, y_baseline, multioutput='raw_values')
mse_baseline

array([3.51455738e+03, 4.12706217e+00, 5.50119226e-02, 2.06390099e-01,
       1.66484681e+02, 4.71277124e+02])

In [21]:
classic_models_dir = Path("./models/hsi")
classic_results = {}

for model_path in classic_models_dir.iterdir():
    if model_path.name.startswith("ebm"):
        continue
        
    if model_path.is_file():
        model = MultiOutputTuner.load(model_path)
        y_pred = model.predict(X_train)

        mse = mean_squared_error(y_train,y_pred, multioutput='raw_values')
        classic_results[model_path.name] = mse/mse_baseline
df_classic_results = pd.DataFrame(classic_results).T
df_classic_results.index.name = "model"
print(column_names)
df_classic_results

['Fe', 'Zn', 'B', 'Cu', 'S', 'Mn']


,0,1,2,3,4,5
model,,,,,,
gradientBoosting,0.404159,0.317792,0.368343,0.479734,0.462124,0.464421
kNeighbors,0.442626,0.384154,0.402718,0.441975,0.401692,0.395866
adaBoost,0.721241,0.766127,0.712064,0.837783,0.760669,0.700646
gradientBoosting_95,0.718645,0.874904,0.758972,0.859713,0.733984,0.759633
xgb,0.416040,0.308843,0.577134,0.528808,0.406467,0.329684
xgb_95,0.744248,0.762981,0.756419,0.853232,0.785052,0.755086
kNeighbors_95,0.754000,0.826905,0.769930,0.857200,0.811943,0.427419
adaBoost_95,0.754134,0.859632,0.779466,0.876514,0.800472,0.781339
xgb_975,0.715136,0.828904,0.759398,0.849009,0.784504,0.694284


CNN2D 0.95 =     0.53048769, 0.66062731, 0.50172776, 0.72228935, 0.65769953, 0.58475807
CNN3D 0.95 =     0.49918575, 0.62049721, 0.45977483, 0.68627737, 0.61806255, 0.53619069
CNN2D =          0.46976451, 0.5716636 , 0.42982737, 0.66206401, 0.55735863, 0.49101401
RNN  0.95=       0.4231152 , 0.41313581, 0.37148547, 0.51408543, 0.5350001, 0.55188089
RNN =            0.42545757, 0.47854947, 0.38916031, 0.59854028, 0.56769386, 0.52527768
BandRNN =        0.9998284 , 1.00009714, 1.00006329, 1.0000605 , 0.99993886, 0.99997799
BandRNN 0.95 =   0.49994122, 0.67954018, 0.52331478, 0.74543027, 0.66064053, 0.57780528
FusionRNN 0.95 = 0.3984208 , 0.41592714, 0.30646235, 0.49188126, 0.50427348, 0.4238029
FusionRNN =      0.39059347, 0.39539525, 0.29104869, 0.45612823, 0.49257277, 0.40915999

# Final training model

In [ ]:
from neural_networks.Recurrent import FusionNN

# For each particle Fusion recurrent neural network gave the best estimation
fe_model = FusionNN(neurons=256, hidden_size=12000).to(device)
fe_model = torch.load_state_dict(torch.load("models/nn/FusionNN.pt")

# Final testing project